# Task 4 — DINOv3 ViT-B/16 (5-fold OOF + test predictions)

Fine-tune one pretrained image classifier with a fixed five-fold stratified split using both T4 GPUs. Each fold model predicts its held-out training fold and the test set; the five test probability matrices are averaged. The notebook writes OOF scores and a separate test prediction CSV for local ensembling.

Data source: `microhum/cpe333-task-4-dataset`. CSV columns are `id,file_name,label`; images are loaded by `file_name`. The test CSV is read with only `id,file_name` columns; test labels are never loaded or used. Class order: 7-11 Knights, FiveN, Free Fried, PUBE, Roblock.

Model: `vit_base_patch16_dinov3.lvd1689m`. DINOv3 weights require accepting Meta's DINOv3 license and Hugging Face account access. Add Kaggle Secrets HF_USERNAME and HUGGING_FACE_HUB_TOKEN; the token is not printed.

Artifacts in `/kaggle/working/`: `task4-dinov3-vitb-oof.csv`, `task4-dinov3-vitb-test-predictions.csv`, and an OOF JSON summary.

The worker probes the largest tested global batch size that fits both T4 GPUs with a 10% memory margin; the chosen size is recorded in the log and manifest. Best-epoch model weights are saved as one inference checkpoint per fold; the worker manifest and all-artifacts ZIP include these checkpoints. The OOF/test probability CSVs, epoch metrics, manifest, recovery arrays, log, worker source, confusion-matrix PNG, prediction distribution, and an all-artifacts ZIP are saved under `/kaggle/working/`. Post-ensemble only needs the test CSV `id` plus `prob_0`–`prob_4` columns; weights are not required. Use Kaggle Save Version after the run to preserve notebook outputs beyond the ephemeral runtime.


In [ ]:
%pip uninstall -y timm
%pip install --pre "timm>=1.0.20" huggingface_hub

import os
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import timm
import torch
from huggingface_hub import HfApi, login
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
HF_USERNAME = secrets.get_secret("HF_USERNAME").strip()
HF_TOKEN = secrets.get_secret("HUGGING_FACE_HUB_TOKEN").strip()
assert HF_USERNAME and HF_TOKEN, (
    "Add HF_USERNAME and HUGGING_FACE_HUB_TOKEN in Kaggle Secrets."
)
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=False)
identity = HfApi().whoami(token=HF_TOKEN)
assert identity.get("name", "").casefold() == HF_USERNAME.casefold(), (
    "HF_USERNAME does not match the account for HUGGING_FACE_HUB_TOKEN."
)
print("Hugging Face account verified:", HF_USERNAME, "(token hidden)")
print("timm", timm.__version__, "torch", torch.__version__)
print("GPUs", torch.cuda.device_count())
subprocess.run(["nvidia-smi"], check=True)

In [ ]:
from IPython.display import display

INPUT_DIR = Path("/kaggle/input/datasets/microhum/cpe333-task-4-dataset")
TRAIN_CSV = INPUT_DIR / "train.csv"
TEST_CSV = INPUT_DIR / "test.csv"
TRAIN_IMAGES = INPUT_DIR / "train"
TEST_IMAGES = INPUT_DIR / "test"
OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert TRAIN_CSV.is_file(), f"Missing official training CSV: {TRAIN_CSV}"
assert TEST_CSV.is_file(), f"Missing official test CSV: {TEST_CSV}"
assert TRAIN_IMAGES.is_dir() and TEST_IMAGES.is_dir()
assert torch.cuda.is_available() and torch.cuda.device_count() >= 2, (
    "Expected dual T4 GPUs."
)

train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV, usecols=["id", "file_name"])
assert {"id", "file_name", "label"}.issubset(train.columns)
assert {"id", "file_name"}.issubset(test.columns)
assert train["id"].is_unique and test["id"].is_unique
assert train["file_name"].is_unique and test["file_name"].is_unique
CLASS_NAMES = ["7-11 Knights", "FiveN", "Free Fried", "PUBE", "Roblock"]
train["label"] = pd.to_numeric(train["label"], errors="raise").astype(int)
assert train["label"].between(0, len(CLASS_NAMES) - 1).all()

# Sample a few paths; avoid slow serial stat calls across the entire Kaggle mount.
for image_dir, names in (
    (TRAIN_IMAGES, train["file_name"]),
    (TEST_IMAGES, test["file_name"]),
):
    missing = [
        name for name in names.astype(str).head(5) if not (image_dir / name).is_file()
    ]
    assert not missing, f"Sample image files missing in {image_dir}: {missing}"

print(f"Train rows: {len(train):,}; test rows: {len(test):,}")
display(
    train["label"]
    .value_counts()
    .sort_index()
    .rename(index=dict(enumerate(CLASS_NAMES)))
    .rename("count")
    .to_frame()
)
print("Test labels are not loaded; test is used only to make predictions.")
print("GPUs:", torch.cuda.get_device_name(0), "and", torch.cuda.get_device_name(1))

In [ ]:
WORKER_SOURCE = r"""
import argparse
import gc
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import timm
import torch
from PIL import Image, ImageFile
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

ImageFile.LOAD_TRUNCATED_IMAGES = True
CLASS_NAMES = ["7-11 Knights", "FiveN", "Free Fried", "PUBE", "Roblock"]
BATCH_CANDIDATES = (2, 4, 8, 12, 16, 20, 24, 32, 40, 48)


def atomic_json(path, payload):
    path = Path(path)
    temp_path = path.with_suffix(path.suffix + ".tmp")
    temp_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    temp_path.replace(path)


def make_transform(training, mean, std):
    ops = []
    if training:
        ops.extend([
            transforms.RandomAffine(degrees=2, translate=(0.02, 0.02), scale=(0.98, 1.02)),
            transforms.ColorJitter(brightness=0.10, contrast=0.10, saturation=0.05),
        ])
    ops.extend([transforms.ToTensor(), transforms.Normalize(mean, std)])
    return transforms.Compose(ops)


class ScreenshotDataset(Dataset):
    def __init__(self, frame, image_dir, training, image_size, mean, std):
        self.file_names = frame["file_name"].astype(str).tolist()
        self.labels = frame["label"].to_numpy(dtype=np.int64) if "label" in frame else None
        self.image_dir = Path(image_dir)
        self.image_size = image_size
        self.transform = make_transform(training, mean, std)

    def __len__(self):
        return len(self.file_names)

    def __getitem__(self, index):
        image_path = self.image_dir / self.file_names[index]
        with Image.open(image_path) as image:
            image = image.convert("RGB")
            image.thumbnail((self.image_size, self.image_size), Image.Resampling.LANCZOS)
            canvas = Image.new("RGB", (self.image_size, self.image_size), (0, 0, 0))
            canvas.paste(
                image,
                ((self.image_size - image.width) // 2, (self.image_size - image.height) // 2),
            )
        image_tensor = self.transform(canvas)
        if self.labels is None:
            return image_tensor
        return image_tensor, int(self.labels[index])


def make_loader(dataset, training, batch_size):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=training,
        num_workers=4,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=2,
    )


def choose_batch_size(model_name, device):
    probe = timm.create_model(model_name, pretrained=True, num_classes=len(CLASS_NAMES))
    data_config = timm.data.resolve_model_data_config(probe)
    image_size = int(data_config["input_size"][-1])
    mean, std = data_config["mean"], data_config["std"]
    probe = nn.DataParallel(probe.to(device), device_ids=[0, 1])
    initial_state = {
        key: value.detach().cpu().clone()
        for key, value in probe.module.state_dict().items()
    }
    best_batch_size = None
    best_peak_memory = None
    safe_limit = 0.90

    for candidate in BATCH_CANDIDATES:
        optimizer = None
        scaler = None
        images = None
        targets = None
        oom = False
        try:
            probe.module.load_state_dict(initial_state)
            optimizer = torch.optim.AdamW(probe.parameters(), lr=2e-5, weight_decay=1e-4)
            scaler = torch.amp.GradScaler("cuda")
            images = torch.randn(
                candidate, 3, image_size, image_size, device=device, dtype=torch.float32
            )
            targets = torch.randint(len(CLASS_NAMES), (candidate,), device=device)
            probe.train()
            optimizer.zero_grad(set_to_none=True)
            for gpu_index in range(2):
                torch.cuda.reset_peak_memory_stats(gpu_index)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = probe(images)
                loss = nn.functional.cross_entropy(logits, targets)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            torch.cuda.synchronize()
            peaks = [torch.cuda.max_memory_allocated(index) for index in range(2)]
            totals = [torch.cuda.get_device_properties(index).total_memory for index in range(2)]
            fractions = [peak / total for peak, total in zip(peaks, totals)]
            peak_gb = [round(peak / 1024**3, 2) for peak in peaks]
            print(
                f"batch probe={candidate} per-GPU peak={peak_gb} GiB "
                f"usage={[round(value * 100, 1) for value in fractions]}%",
                flush=True,
            )
            if max(fractions) <= safe_limit:
                best_batch_size = candidate
                best_peak_memory = peak_gb
            elif best_batch_size is None:
                best_batch_size = candidate
                best_peak_memory = peak_gb
                print("No tested batch fits the 90% memory target; keeping the smallest fit.", flush=True)
                break
            else:
                break
        except RuntimeError as error:
            if "out of memory" not in str(error).lower():
                raise
            oom = True
            print(f"batch probe={candidate} hit CUDA OOM; using the previous safe size.", flush=True)
        finally:
            if optimizer is not None:
                optimizer.zero_grad(set_to_none=True)
            del optimizer, scaler, images, targets
            gc.collect()
            torch.cuda.empty_cache()
        if oom:
            break

    del probe, initial_state
    gc.collect()
    torch.cuda.empty_cache()
    if best_batch_size is None:
        raise RuntimeError("No dual-GPU training batch size fit; lower the model input resolution.")
    print(
        f"Selected global batch_size={best_batch_size}; last accepted per-GPU peaks="
        f"{best_peak_memory} GiB. This is a memory-based probe, not a utilization guarantee.",
        flush=True,
    )
    return image_size, list(mean), list(std), best_batch_size


def predict_probabilities(model, loader, device, has_labels):
    probabilities = []
    model.eval()
    with torch.inference_mode():
        for batch in loader:
            images = batch[0] if has_labels else batch
            images = images.to(device, non_blocking=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = model(images)
            probabilities.append(logits.float().softmax(dim=1).cpu().numpy())
    return np.concatenate(probabilities)


def atomic_csv(frame, path):
    path = Path(path)
    temp_path = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temp_path, index=False)
    temp_path.replace(path)


def save_recovery(path, oof, fold_ids, test_probability_sum, completed_folds):
    path = Path(path)
    temp_path = path.with_suffix(path.suffix + ".tmp")
    with temp_path.open("wb") as file_obj:
        np.savez_compressed(
            file_obj,
            oof=oof,
            fold_ids=fold_ids,
            test_probability_sum=test_probability_sum,
            completed_folds=np.asarray(completed_folds, dtype=np.int8),
        )
    temp_path.replace(path)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model-name", required=True)
    parser.add_argument("--input-dir", required=True)
    parser.add_argument("--oof-output", required=True)
    parser.add_argument("--test-output", required=True)
    parser.add_argument("--metrics-output", required=True)
    parser.add_argument("--manifest-output", required=True)
    parser.add_argument("--recovery-output", required=True)
    parser.add_argument("--epochs", type=int, default=5)
    parser.add_argument("--folds", type=int, default=5)
    args = parser.parse_args()

    assert torch.cuda.is_available() and torch.cuda.device_count() >= 2, "Expected dual T4 GPUs."
    device = torch.device("cuda:0")
    print(
        f"model={args.model_name}; GPUs={torch.cuda.get_device_name(0)}, "
        f"{torch.cuda.get_device_name(1)}; folds={args.folds}; epochs={args.epochs}",
        flush=True,
    )

    input_dir = Path(args.input_dir)
    train = pd.read_csv(input_dir / "train.csv")
    # Deliberately exclude the label column: test labels are never read.
    test = pd.read_csv(input_dir / "test.csv", usecols=["id", "file_name"])
    assert {"id", "file_name", "label"}.issubset(train.columns)
    assert {"id", "file_name"}.issubset(test.columns)
    assert train["id"].is_unique and test["id"].is_unique
    assert train["file_name"].is_unique and test["file_name"].is_unique
    train["label"] = pd.to_numeric(train["label"], errors="raise").astype(int)
    assert train["label"].between(0, len(CLASS_NAMES) - 1).all()

    labels = train["label"].to_numpy(dtype=np.int64)
    train_ids = train["id"].astype(str).to_numpy()
    train_files = train["file_name"].astype(str).to_numpy()
    test_ids = test["id"].astype(str).to_numpy()
    test_files = test["file_name"].astype(str).to_numpy()
    image_size, mean, std, batch_size = choose_batch_size(args.model_name, device)
    oof = np.full((len(train), len(CLASS_NAMES)), np.nan, dtype=np.float32)
    fold_ids = np.full(len(train), -1, dtype=np.int8)
    test_probability_sum = np.zeros((len(test), len(CLASS_NAMES)), dtype=np.float64)
    epoch_rows = []
    fold_results = []
    checkpoint_paths = []
    splitter = StratifiedKFold(n_splits=args.folds, shuffle=True, random_state=42)
    manifest_path = Path(args.manifest_output)
    metrics_path = Path(args.metrics_output)
    recovery_path = Path(args.recovery_output)
    atomic_json(manifest_path, {
        "status": "running",
        "model_name": args.model_name,
        "class_names": CLASS_NAMES,
        "folds": args.folds,
        "epochs_per_fold": args.epochs,
        "batch_size": batch_size,
        "image_size": image_size,
        "train_rows": len(train),
        "test_rows": len(test),
        "test_labels_loaded": False,
        "checkpoint_files_saved": False,
        "completed_folds": [],
    })

    for fold, (train_idx, valid_idx) in enumerate(splitter.split(np.zeros(len(labels)), labels)):
        fold_number = fold + 1
        seed = 42 + fold
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        train_frame = train.iloc[train_idx].reset_index(drop=True)
        valid_frame = train.iloc[valid_idx].reset_index(drop=True)
        class_counts = np.bincount(labels[train_idx], minlength=len(CLASS_NAMES))
        class_weights = len(train_idx) / (len(CLASS_NAMES) * class_counts)
        criterion = nn.CrossEntropyLoss(
            weight=torch.tensor(class_weights, dtype=torch.float32, device=device),
            label_smoothing=0.03,
        )

        model = timm.create_model(args.model_name, pretrained=True, num_classes=len(CLASS_NAMES))
        train_ds = ScreenshotDataset(train_frame, input_dir / "train", True, image_size, mean, std)
        valid_ds = ScreenshotDataset(valid_frame, input_dir / "train", False, image_size, mean, std)
        test_ds = ScreenshotDataset(test, input_dir / "test", False, image_size, mean, std)
        train_loader = make_loader(train_ds, True, batch_size)
        valid_loader = make_loader(valid_ds, False, batch_size)
        test_loader = make_loader(test_ds, False, batch_size)

        model = nn.DataParallel(model.to(device), device_ids=[0, 1])
        optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)
        scaler = torch.amp.GradScaler("cuda")
        best_macro_f1 = -1.0
        best_epoch = 0
        best_valid_probabilities = None
        best_model_state = None

        for epoch in range(1, args.epochs + 1):
            model.train()
            batch_losses = []
            epoch_start = time.time()
            for images, targets in train_loader:
                images = images.to(device, non_blocking=True)
                targets = targets.to(device, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    logits = model(images)
                    loss = criterion(logits, targets)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                batch_losses.append(loss.detach())
            scheduler.step()

            valid_probabilities = predict_probabilities(model, valid_loader, device, has_labels=True)
            valid_predictions = valid_probabilities.argmax(axis=1)
            valid_macro_f1 = float(f1_score(labels[valid_idx], valid_predictions, average="macro"))
            valid_weighted_f1 = float(f1_score(labels[valid_idx], valid_predictions, average="weighted"))
            mean_loss = float(torch.stack(batch_losses).mean().item())
            metric = {
                "fold": fold_number,
                "epoch": epoch,
                "train_loss": mean_loss,
                "val_macro_f1": valid_macro_f1,
                "val_weighted_f1": valid_weighted_f1,
                "seconds": time.time() - epoch_start,
            }
            epoch_rows.append(metric)
            atomic_csv(pd.DataFrame(epoch_rows), metrics_path)

            if valid_macro_f1 > best_macro_f1:
                best_macro_f1 = valid_macro_f1
                best_epoch = epoch
                best_valid_probabilities = valid_probabilities.copy()
                best_model_state = {
                    key: value.detach().cpu().clone()
                    for key, value in model.module.state_dict().items()
                }

            print(
                f"fold={fold_number}/{args.folds} epoch={epoch}/{args.epochs} "
                f"batch={batch_size} loss={mean_loss:.4f} val_macro_f1={valid_macro_f1:.6f} "
                f"best={best_macro_f1:.6f}@{best_epoch} seconds={metric['seconds']:.1f}",
                flush=True,
            )

        assert best_valid_probabilities is not None and best_model_state is not None
        model.module.load_state_dict(best_model_state)
        checkpoint_path = manifest_path.with_name(
            f"{manifest_path.stem}-fold-{fold_number}-best.pt"
        )
        checkpoint_temp = checkpoint_path.with_suffix(checkpoint_path.suffix + ".tmp")
        torch.save({
            "model_name": args.model_name,
            "fold": fold_number,
            "best_epoch": best_epoch,
            "best_val_macro_f1": best_macro_f1,
            "state_dict": best_model_state,
        }, checkpoint_temp)
        checkpoint_temp.replace(checkpoint_path)
        checkpoint_paths.append(str(checkpoint_path))
        oof[valid_idx] = best_valid_probabilities
        fold_ids[valid_idx] = fold
        fold_test_probabilities = predict_probabilities(model, test_loader, device, has_labels=False)
        test_probability_sum += fold_test_probabilities
        fold_result = {
            "fold": fold_number,
            "seed": seed,
            "best_epoch": best_epoch,
            "best_val_macro_f1": best_macro_f1,
            "best_val_weighted_f1": float(f1_score(
                labels[valid_idx], best_valid_probabilities.argmax(axis=1), average="weighted"
            )),
        }
        fold_results.append(fold_result)
        save_recovery(recovery_path, oof, fold_ids, test_probability_sum, [x["fold"] for x in fold_results])
        atomic_json(manifest_path, {
            "status": "running",
            "model_name": args.model_name,
            "class_names": CLASS_NAMES,
            "folds": args.folds,
            "epochs_per_fold": args.epochs,
            "batch_size": batch_size,
            "image_size": image_size,
            "train_rows": len(train),
            "test_rows": len(test),
            "test_labels_loaded": False,
            "checkpoint_files_saved": len(checkpoint_paths) == len(fold_results),
            "checkpoint_paths": checkpoint_paths,
            "completed_folds": fold_results,
            "metrics_path": str(metrics_path),
            "recovery_path": str(recovery_path),
        })
        print(f"Fold {fold_number} predictions and recovery arrays saved.", flush=True)
        del model, optimizer, scheduler, scaler, train_loader, valid_loader, test_loader
        del train_ds, valid_ds, test_ds, criterion, best_model_state, best_valid_probabilities
        gc.collect()
        torch.cuda.empty_cache()

    assert np.isfinite(oof).all() and (fold_ids >= 0).all()
    test_probabilities = (test_probability_sum / args.folds).astype(np.float32)
    assert np.isfinite(test_probabilities).all()
    assert np.allclose(test_probabilities.sum(axis=1), 1.0, atol=1e-3)

    oof_frame = pd.DataFrame({
        "id": train_ids,
        "file_name": train_files,
        "label": labels,
        "fold": fold_ids,
    })
    for class_idx in range(len(CLASS_NAMES)):
        oof_frame[f"prob_{class_idx}"] = oof[:, class_idx]
    atomic_csv(oof_frame, args.oof_output)

    test_frame = pd.DataFrame({"id": test_ids, "file_name": test_files})
    test_frame["predicted_label"] = test_probabilities.argmax(axis=1).astype(int)
    test_frame["predicted_class"] = [CLASS_NAMES[i] for i in test_frame["predicted_label"]]
    for class_idx in range(len(CLASS_NAMES)):
        test_frame[f"prob_{class_idx}"] = test_probabilities[:, class_idx]
    atomic_csv(test_frame, args.test_output)
    summary = {
        "status": "complete",
        "model_name": args.model_name,
        "class_names": CLASS_NAMES,
        "folds": args.folds,
        "epochs_per_fold": args.epochs,
        "batch_size": batch_size,
        "image_size": image_size,
        "train_rows": len(train),
        "test_rows": len(test),
        "test_labels_loaded": False,
        "checkpoint_files_saved": len(checkpoint_paths) == args.folds,
        "checkpoint_paths": checkpoint_paths,
        "oof_macro_f1": float(f1_score(labels, oof.argmax(axis=1), average="macro")),
        "oof_weighted_f1": float(f1_score(labels, oof.argmax(axis=1), average="weighted")),
        "oof_path": args.oof_output,
        "test_predictions_path": args.test_output,
        "metrics_path": str(metrics_path),
        "manifest_path": str(manifest_path),
        "recovery_path": str(recovery_path),
        "fold_results": fold_results,
    }
    atomic_json(manifest_path, summary)
    atomic_json(Path(args.oof_output).with_suffix(".json"), summary)
    print(json.dumps(summary, indent=2), flush=True)


if __name__ == "__main__":
    main()

"""
print("Worker saves best model inference checkpoints per fold.")

In [ ]:
MODEL_NAME = "vit_base_patch16_dinov3.lvd1689m"
RUN_SLUG = "task4-dinov3-vitb"
OOF_PATH = OUTPUT_DIR / f"{RUN_SLUG}-oof.csv"
TEST_OUTPUT_PATH = OUTPUT_DIR / f"{RUN_SLUG}-test-predictions.csv"
FOLD_METRICS_PATH = OUTPUT_DIR / f"{RUN_SLUG}-epoch-metrics.csv"
MANIFEST_PATH = OUTPUT_DIR / f"{RUN_SLUG}-manifest.json"
RECOVERY_PATH = OUTPUT_DIR / f"{RUN_SLUG}-recovery.npz"
LOG_PATH = OUTPUT_DIR / f"{RUN_SLUG}.log"
WORKER_PATH = OUTPUT_DIR / f"{RUN_SLUG}-worker.py"
WORKER_PATH.write_text(WORKER_SOURCE)

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0,1"
env["OMP_NUM_THREADS"] = "1"
env["MKL_NUM_THREADS"] = "1"
command = [
    sys.executable,
    str(WORKER_PATH),
    "--model-name",
    MODEL_NAME,
    "--input-dir",
    str(INPUT_DIR),
    "--oof-output",
    str(OOF_PATH),
    "--test-output",
    str(TEST_OUTPUT_PATH),
    "--metrics-output",
    str(FOLD_METRICS_PATH),
    "--manifest-output",
    str(MANIFEST_PATH),
    "--recovery-output",
    str(RECOVERY_PATH),
    "--epochs",
    "5",
    "--folds",
    "5",
]
with LOG_PATH.open("w") as log_file:
    process = subprocess.Popen(
        command,
        cwd=str(OUTPUT_DIR),
        env=env,
        stdout=log_file,
        stderr=subprocess.STDOUT,
    )
    offset = 0
    while process.poll() is None:
        time.sleep(30)
        with LOG_PATH.open("r") as current_log:
            current_log.seek(offset)
            new_text = current_log.read()
            offset = current_log.tell()
        if new_text:
            print(new_text, end="", flush=True)
    return_code = process.wait()
if return_code != 0:
    raise RuntimeError(
        f"Worker failed (exit={return_code}). Log tail:\n"
        f"{LOG_PATH.read_text(errors='replace')[-8000:]}"
    )
print("Completed OOF:", OOF_PATH)
print("Completed test probabilities for post-ensemble:", TEST_OUTPUT_PATH)
print("GPU memory probe selected the batch size; see the log and manifest.")

In [ ]:
import zipfile

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, f1_score

OOF_RESULT = pd.read_csv(OOF_PATH)
assert OOF_RESULT["id"].astype(str).tolist() == train["id"].astype(str).tolist()
assert (
    OOF_RESULT["file_name"].astype(str).tolist()
    == train["file_name"].astype(str).tolist()
)
assert OOF_RESULT["label"].tolist() == train["label"].tolist()
oof_probabilities = OOF_RESULT[
    [f"prob_{i}" for i in range(len(CLASS_NAMES))]
].to_numpy()
assert np.isfinite(oof_probabilities).all() and (oof_probabilities >= 0).all()
assert np.allclose(oof_probabilities.sum(axis=1), 1.0, atol=1e-3)
oof_predictions = oof_probabilities.argmax(axis=1)
print("OOF macro F1:", f1_score(OOF_RESULT["label"], oof_predictions, average="macro"))
print(
    "OOF weighted F1:",
    f1_score(OOF_RESULT["label"], oof_predictions, average="weighted"),
)
print(
    classification_report(
        OOF_RESULT["label"],
        oof_predictions,
        labels=range(len(CLASS_NAMES)),
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0,
    )
)

matrix = confusion_matrix(
    OOF_RESULT["label"], oof_predictions, labels=range(len(CLASS_NAMES))
)
CONFUSION_PATH = OUTPUT_DIR / f"{RUN_SLUG}-oof-confusion-matrix.png"
fig, axis = plt.subplots(figsize=(7, 5))
sns.heatmap(
    matrix,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
    ax=axis,
)
axis.set(
    title="DINOv3 ViT-B/16 — best-epoch OOF confusion matrix",
    xlabel="Predicted",
    ylabel="Actual",
)
plt.tight_layout()
fig.savefig(CONFUSION_PATH, dpi=300, bbox_inches="tight")
plt.show()

TEST_RESULT = pd.read_csv(TEST_OUTPUT_PATH)
assert TEST_RESULT["id"].astype(str).tolist() == test["id"].astype(str).tolist()
assert (
    TEST_RESULT["file_name"].astype(str).tolist()
    == test["file_name"].astype(str).tolist()
)
assert TEST_RESULT["id"].is_unique
assert TEST_RESULT["predicted_label"].between(0, len(CLASS_NAMES) - 1).all()
test_probabilities = TEST_RESULT[
    [f"prob_{i}" for i in range(len(CLASS_NAMES))]
].to_numpy()
assert np.isfinite(test_probabilities).all() and (test_probabilities >= 0).all()
assert np.allclose(test_probabilities.sum(axis=1), 1.0, atol=1e-3)
assert np.array_equal(
    TEST_RESULT["predicted_label"].to_numpy(), test_probabilities.argmax(axis=1)
)
print(f"Validated test predictions: {len(TEST_RESULT):,} rows")
print(TEST_RESULT["predicted_class"].value_counts())
DISTRIBUTION_PATH = OUTPUT_DIR / f"{RUN_SLUG}-test-prediction-distribution.csv"
TEST_RESULT["predicted_class"].value_counts().rename_axis(
    "predicted_class"
).reset_index(name="count").to_csv(DISTRIBUTION_PATH, index=False)

ARTIFACT_ZIP = OUTPUT_DIR / f"{RUN_SLUG}-all-artifacts.zip"
checkpoint_paths = [
    OUTPUT_DIR / f"{MANIFEST_PATH.stem}-fold-{fold}-best.pt" for fold in range(1, 6)
]
artifact_paths = [
    OOF_PATH,
    OOF_PATH.with_suffix(".json"),
    TEST_OUTPUT_PATH,
    FOLD_METRICS_PATH,
    MANIFEST_PATH,
    RECOVERY_PATH,
    *checkpoint_paths,
    LOG_PATH,
    WORKER_PATH,
    CONFUSION_PATH,
    DISTRIBUTION_PATH,
]
missing_artifacts = [str(path) for path in artifact_paths if not path.is_file()]
assert not missing_artifacts, f"Expected run artifacts missing: {missing_artifacts}"
with zipfile.ZipFile(
    ARTIFACT_ZIP, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1
) as archive:
    for path in artifact_paths:
        archive.write(path, arcname=path.relative_to(OUTPUT_DIR))
print("All artifacts bundle:", ARTIFACT_ZIP)
print(
    "All five fold inference checkpoints were saved with the prediction and recovery artifacts."
)
print(
    "Use Kaggle Save Version after the run, or download this ZIP before the runtime ends."
)